# LangGraph Agentic Workflow: Product Description Generator

This notebook demonstrates how to build an **agentic workflow** using LangGraph.

Instead of asking one LLM prompt to perform the entire task, we divide the work into smaller steps. Each step performs one responsibility, stores its result in a shared state, and passes control to the next step.

## What students will learn

- What LangGraph is and why it is useful
- How **state** carries data through a workflow
- How **nodes** perform individual tasks
- How **edges** control the direction of execution
- How to build sequential, parallel, conditional, and looping workflows
- How multiple AI steps can work together like a team of specialized agents


In [2]:
!pip install -U langgraph langchain langchain-core langchain-community \
               langchain-openai langchain-experimental \
               pydantic typing-extensions


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# -----------------------------------------
# Imports
# -----------------------------------------
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

In [4]:
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [5]:
# -----------------------------------------
# LLM (OpenRouter)
# -----------------------------------------
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens = 300,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)


## 1. Shared State: The Workflow's Memory

A LangGraph workflow needs a common data structure called **state**.

Think of the state as a shared notebook carried through the graph. Every node can:

1. Read information already stored in the state
2. Perform its task
3. Return one or more updated state values

In this example, the state begins with a product name. As the graph runs, each node adds new information such as the basic description, features, marketing message, and final description.

```text
Initial State
    ↓
product_name = "Smart Water Bottle"
    ↓
Nodes gradually fill the remaining fields
```

`TypedDict` describes which keys are expected in the state and the type of value stored under each key.


# Dynamic Orchestration with LangGraph
## Product Description Generator

This standalone notebook focuses on **dynamic orchestration** using the same product-description example: **AI-Powered Smart Water Bottle**.

Instead of forcing every product through the same fixed sequence, a planner decides **at runtime** which specialist tasks are useful.

```text
START → Basic Description → Planner → Dynamic Specialists → Synthesizer → END
                                  ├→ features
                                  ├→ audience
                                  ├→ seo
                                  └→ technical
```

### Learning objectives
- Understand dynamic orchestration and runtime planning.
- Use LangGraph `Send` for dynamic fan-out.
- Reuse one specialist worker for multiple task types.
- Aggregate dynamic worker results with a reducer.
- Converge dynamic work into a final synthesizer.
- Run the same graph against different products.

## 1. Fixed vs Dynamic Orchestration

### Fixed orchestration

```text
basic → features → marketing → final
```

The route is known before execution and every input follows the same path.

### Dynamic orchestration

```text
basic → planner → {features, audience, seo, technical, compliance} → final
```

The planner chooses a subset of capabilities based on the current state.

The graph contains reusable worker capability, while the **execution plan is created at runtime**.

In [6]:
# ============================================================
# 2. Dynamic Workflow State
# ============================================================

from typing import Annotated, TypedDict
import operator
from langgraph.types import Send


class DynamicState(TypedDict):
    product_name: str
    basic_description: str
    requested_tasks: list[str]
    specialist_results: Annotated[list[str], operator.add]
    final_description: str


class SpecialistState(TypedDict):
    product_name: str
    basic_description: str
    task: str
    specialist_results: Annotated[list[str], operator.add]

## 2. Why the Reducer Is Required

Multiple dynamic workers may write results at approximately the same stage:

```text
features ───────┐
audience ────────┤
seo ────────────┼→ specialist_results
technical ─────┘
```

`Annotated[list[str], operator.add]` tells LangGraph to **merge list updates** rather than replace the previous result.

In [7]:
# ============================================================
# 3. Basic Product Description Node
# ============================================================

def generate_basic_description(state: DynamicState):
    response = llm.invoke([
        SystemMessage(content=(
            "Generate concise, factual product descriptions. "
            "Do not invent precise specifications."
        )),
        HumanMessage(content=f"""
Create a concise product description for:

Product: {state['product_name']}

Return 2-4 sentences.
""")
    ])
    return {"basic_description": response.content}

## 3. Planner / Orchestrator

The planner has one responsibility:

> **Decide what specialist work is needed for this product.**

Available tasks:

| Task | Responsibility |
|---|---|
| `features` | Features and customer benefits |
| `audience` | Target customers and motivations |
| `seo` | Search keywords and intent |
| `technical` | Technical capabilities and differentiators |
| `compliance` | Careful handling of product claims |

For one input the planner may return:

```python
["features", "audience", "seo"]
```

For another it may return:

```python
["features", "audience", "seo", "technical", "compliance"]
```

In [8]:
# ============================================================
# 4. Dynamic Planner
# ============================================================

ALLOWED_TASKS = {
    "features", "audience", "seo", "technical", "compliance"
}

def dynamic_planner(state: DynamicState):
    prompt = f"""
You are the runtime orchestrator for a product-description system.

Product:
{state['product_name']}

Basic description:
{state['basic_description']}

Available specialist capabilities:
- features: product features and customer benefits
- audience: target audience and buying motivations
- seo: SEO keywords and search intent
- technical: technical capabilities and differentiators
- compliance: potentially risky claims and careful wording

Choose only the capabilities that are genuinely useful.
Return ONLY a comma-separated list using the exact task names.
Select at least 2 and no more than 5 tasks.
Example: features,audience,seo
"""

    response = llm.invoke([
        SystemMessage(content="You are a precise workflow planner."),
        HumanMessage(content=prompt)
    ])

    tasks = [
        x.strip().lower()
        for x in response.content.split(",")
        if x.strip().lower() in ALLOWED_TASKS
    ]
    tasks = list(dict.fromkeys(tasks))

    if len(tasks) < 2:
        tasks = ["features", "audience", "seo"]

    print("Runtime execution plan:", tasks)
    return {"requested_tasks": tasks}

## 4. Dynamic Fan-Out with `Send`

A normal edge is static:

```python
A → B
```

`Send` allows a node to create runtime work items:

```python
Send("specialist", {"task": "features"})
Send("specialist", {"task": "seo"})
Send("specialist", {"task": "audience"})
```

Therefore the number of specialist executions can depend on the planner output.

```text
                    ┌→ specialist(features)
planner ────────────┼→ specialist(audience)
                    └→ specialist(seo)
```

In [9]:
# ============================================================
# 5. Dynamic Dispatcher
# ============================================================

def dispatch_specialists(state: DynamicState):
    return [
        Send(
            "specialist",
            {
                "product_name": state["product_name"],
                "basic_description": state["basic_description"],
                "task": task,
                "specialist_results": [],
            },
        )
        for task in state["requested_tasks"]
    ]

## 5. Reusable Specialist Worker

Instead of creating five hard-coded graph nodes, we use one generic worker:

```text
specialist
```

The task is carried in the dynamic state:

```python
state["task"]
```

The planner decides **what** to do; the specialist worker performs **how** to do it.

In [10]:
# ============================================================
# 6. Generic Specialist Worker
# ============================================================

TASK_INSTRUCTIONS = {
    "features": "Identify important features and customer benefits.",
    "audience": "Identify target customers, needs, pain points, and motivations.",
    "seo": "Identify useful SEO keywords and search-intent phrases.",
    "technical": "Identify relevant technical capabilities and differentiators. Do not invent specifications.",
    "compliance": "Identify claims that need careful wording and recommend accurate language.",
}


def run_specialist(state: SpecialistState):
    task = state["task"]
    response = llm.invoke([
        SystemMessage(content="You are a specialist analyst supporting product-copy generation."),
        HumanMessage(content=f"""
Product:
{state['product_name']}

Basic description:
{state['basic_description']}

Specialist task:
{task}

Instruction:
{TASK_INSTRUCTIONS[task]}

Return concise research that a final product writer can use.
""")
    ])

    return {
        "specialist_results": [
            f"===== {task.upper()} SPECIALIST =====\n{response.content}"
        ]
    }

## 6. Synthesizer / Convergence

All dynamically generated specialist results are collected and passed to one final node.

```text
features ───────┐
audience ────────┤
seo ────────────┼→ specialist_results → synthesizer → final
technical ──────┤
compliance ─────┘
```

The synthesizer does not need to know the plan in advance. It simply consumes the research that was actually produced.

In [11]:
# ============================================================
# 7. Final Synthesizer
# ============================================================

def synthesize_dynamic_result(state: DynamicState):
    research = "\n\n".join(state["specialist_results"])

    response = llm.invoke([
        SystemMessage(content=(
            "You are a senior product-copy writer. "
            "Create accurate, customer-focused product copy."
        )),
        HumanMessage(content=f"""
Product:
{state['product_name']}

Basic description:
{state['basic_description']}

Specialist research selected dynamically:
{research}

Create the final product description.
Do not invent unsupported specifications.
Communicate customer value clearly.
""")
    ])

    return {"final_description": response.content}

## 7. Build the Dynamic Graph

The graph is:

```text
START
  ↓
Basic
  ↓
Planner
  ↓
Dynamic Dispatcher
  ↓
┌─────────┬─────────┬─────────┐
Features  Audience   SEO    Technical...
└─────────┴─────────┴─────────┘
             ↓
        Synthesizer
             ↓
            END
```

The critical operation is:

```python
workflow.add_conditional_edges(
    "planner",
    dispatch_specialists,
    ["specialist"]
)
```

The dispatcher can return a variable number of `Send` objects, so the runtime execution plan is dynamic.

In [12]:
# ============================================================
# 8. Build and Compile
# ============================================================

def build_dynamic_workflow():
    workflow = StateGraph(DynamicState)

    workflow.add_node("basic", generate_basic_description)
    workflow.add_node("planner", dynamic_planner)
    workflow.add_node("specialist", run_specialist)
    workflow.add_node("synthesizer", synthesize_dynamic_result)

    workflow.add_edge(START, "basic")
    workflow.add_edge("basic", "planner")

    workflow.add_conditional_edges(
        "planner",
        dispatch_specialists,
        ["specialist"]
    )

    workflow.add_edge("specialist", "synthesizer")
    workflow.add_edge("synthesizer", END)

    return workflow.compile()


dynamic_app = build_dynamic_workflow()
print("Dynamic workflow compiled successfully.")

Dynamic workflow compiled successfully.


## 8. Execute the Same Example

We now run the workflow for **AI-Powered Smart Water Bottle**.

The initial state contains only the product name. The basic description, plan, specialist research, and final answer are produced by the workflow.

In [13]:
# ============================================================
# 9. Run Example 1
# ============================================================

initial_state: DynamicState = {
    "product_name": "AI-Powered Smart Water Bottle",
    "basic_description": "",
    "requested_tasks": [],
    "specialist_results": [],
    "final_description": "",
}

result = dynamic_app.invoke(initial_state)

print("\n" + "=" * 70)
print("DYNAMIC EXECUTION PLAN")
print("=" * 70)
print(result["requested_tasks"])

print("\n" + "=" * 70)
print("BASIC DESCRIPTION")
print("=" * 70)
print(result["basic_description"])

print("\n" + "=" * 70)
print("SPECIALIST RESULTS")
print("=" * 70)
for item in result["specialist_results"]:
    print("\n" + item)

print("\n" + "=" * 70)
print("FINAL PRODUCT DESCRIPTION")
print("=" * 70)
print(result["final_description"])

Runtime execution plan: ['features', 'audience', 'seo', 'technical', 'compliance']

DYNAMIC EXECUTION PLAN
['features', 'audience', 'seo', 'technical', 'compliance']

BASIC DESCRIPTION
The AI-Powered Smart Water Bottle tracks your hydration habits and reminds you to drink water throughout the day. Equipped with a sleek design and a user-friendly app, it customizes hydration goals based on your activity levels and personal preferences. Stay healthy and hydrated effortlessly with this innovative companion.

SPECIALIST RESULTS

===== FEATURES SPECIALIST =====
### Features and Customer Benefits of the AI-Powered Smart Water Bottle

1. **Hydration Tracking**  
   - **Benefit**: Monitors daily water intake, ensuring users meet their hydration goals and promoting overall health.

2. **Personalized Reminders**  
   - **Benefit**: Sends timely notifications to drink water based on individual activity levels and preferences, helping users stay consistent with hydration.

3. **Sleek Design**  
  

## 9. Inspect the Runtime Decision

The important orchestration information is:

- selected task names
- number of dynamic tasks
- number of aggregated results
- final output

This makes the planner decision observable and easier to debug.

In [14]:
# ============================================================
# 10. Inspect Runtime State
# ============================================================

print("Product:", result["product_name"])
print("\nSelected tasks:", result["requested_tasks"])
print("Number of dynamic tasks:", len(result["requested_tasks"]))
print("Number of specialist results:", len(result["specialist_results"]))

Product: AI-Powered Smart Water Bottle

Selected tasks: ['features', 'audience', 'seo', 'technical', 'compliance']
Number of dynamic tasks: 5
Number of specialist results: 5


## 10. Same Graph, Different Product

Now use a different product without changing the graph.

The architecture is reusable:

```text
                 SAME COMPILED GRAPH
                         │
          ┌──────────────┴──────────────┐
          ↓                             ↓
Smart Water Bottle             Enterprise AI Platform
          ↓                             ↓
       planner                       planner
          ↓                             ↓
    runtime plan                 runtime plan
          ↓                             ↓
       N tasks                       M tasks
          └──────────────┬──────────────┘
                         ↓
                    synthesizer
```

In [15]:
# ============================================================
# 11. Run Example 2
# ============================================================

second_state: DynamicState = {
    "product_name": "Enterprise AI Document Processing Platform",
    "basic_description": "",
    "requested_tasks": [],
    "specialist_results": [],
    "final_description": "",
}

second_result = dynamic_app.invoke(second_state)

print("SECOND PRODUCT PLAN:")
print(second_result["requested_tasks"])

print("\nSECOND PRODUCT FINAL DESCRIPTION:\n")
print(second_result["final_description"])

Runtime execution plan: ['features', 'audience', 'technical', 'compliance']
SECOND PRODUCT PLAN:
['features', 'audience', 'technical', 'compliance']

SECOND PRODUCT FINAL DESCRIPTION:

### Enterprise AI Document Processing Platform

Transform your document management with the **Enterprise AI Document Processing Platform**, designed to streamline the extraction, classification, and analysis of data from both structured and unstructured documents, including invoices, contracts, and reports. This powerful platform harnesses advanced artificial intelligence to enhance operational efficiency, allowing businesses of all sizes to automate repetitive data entry and document processing tasks, potentially reducing processing time by up to 70%.

#### Key Features and Benefits:

1. **Automated Data Extraction**  
   Say goodbye to time-consuming manual data entry. Our platform significantly reduces the effort required for data extraction, enabling your team to focus on higher-value tasks that driv

## 11. Conditional Routing vs Dynamic Fan-Out

These are related but different patterns.

### Conditional routing

The graph chooses between predefined branches:

```text
planner → route A
        → route B
        → route C
```

### Dynamic fan-out with `Send`

The planner can create multiple runtime work items:

```text
planner
 ├→ Send(task 1)
 ├→ Send(task 2)
 └→ Send(task N)
```

This is particularly useful when the number of tasks is not known when the graph is designed.

## 12. Single Responsibility Principle

| Component | Responsibility |
|---|---|
| `generate_basic_description` | Create initial context |
| `dynamic_planner` | Decide required work |
| `dispatch_specialists` | Create runtime tasks |
| `run_specialist` | Execute one selected task |
| `synthesize_dynamic_result` | Create final answer |
| `DynamicState` | Carry workflow data |
| reducer | Merge worker results |

This separation keeps orchestration logic independent from specialist logic and final copy generation.

## 13. General Agentic Pattern

The product-description example generalizes to many agentic systems:

```text
                     ┌→ Agent A ─┐
                     │           │
Input → Planner ──────┼→ Agent B ─┼→ Aggregator → Final Agent
                     │           │
                     ├→ Agent C ─┤
                     │           │
                     └→ Agent N ─┘
```

Examples include:

- resume/candidate analysis
- multi-document analysis
- research agents
- product intelligence
- dynamic data processing
- multi-source investigation
- task decomposition systems

### Core takeaway

**Planner → Dynamic `Send` tasks → Workers → Reducer → Synthesizer**

That is the dynamic orchestration pattern demonstrated in this notebook.

In [ ]:
# ============================================================
# GRADIO UI FOR DYNAMIC LANGGRAPH AGENTIC WORKFLOW
# ============================================================

!pip install -q gradio

import gradio as gr


def run_dynamic_product_agent(product_name):

    if not product_name or not product_name.strip():
        return "", "", "", "", "", "Please enter a product name."

    initial_state: DynamicState = {
        "product_name": product_name.strip(),
        "basic_description": "",
        "requested_tasks": [],
        "specialist_results": [],
        "final_description": ""
    }

    result = dynamic_app.invoke(initial_state)

    selected_tasks = result.get("requested_tasks", [])
    basic_description = result.get("basic_description", "")
    specialist_results = result.get("specialist_results", [])
    final_description = result.get("final_description", "")

    plan_view = "\n".join(
        f"{i}. `{task}`"
        for i, task in enumerate(selected_tasks, 1)
    )

    specialist_view = ""

    for i, item in enumerate(specialist_results, 1):
        specialist_view += f"""
### Specialist {i}

{item}

---
"""

    if not specialist_view:
        specialist_view = "No specialist results generated."

    statistics_view = f"""
### Runtime Statistics

| Metric | Value |
|---|---:|
| Product | {product_name} |
| Dynamic Tasks Selected | {len(selected_tasks)} |
| Specialist Results | {len(specialist_results)} |
| Workflow Status | Completed |
"""

    process_view = f"""
# Dynamic LangGraph Agentic Workflow

## Product

**{product_name}**

---

## 1. Basic Description

{basic_description}

---

## 2. Runtime Planner

### Selected Tasks

{plan_view}

---

## 3. Dynamic Fan-Out

```text
                    ┌→ Specialist
                    │
Basic → Planner ────┼→ Specialist
                    │
                    ├→ Specialist
                    │
                    └→ Specialist
```

**Dynamic tasks:** {len(selected_tasks)}

---

## 4. Specialist Results

{specialist_view}

---

## 5. Reducer / Aggregation

**Aggregated specialist results:** {len(specialist_results)}

---

## 6. Synthesizer

The dynamically generated specialist research was passed to the final synthesizer.

---

## 7. Final Product Description

{final_description}
"""

    return (
        plan_view,
        basic_description,
        specialist_view,
        statistics_view,
        final_description,
        process_view
    )


with gr.Blocks(
    title="Dynamic LangGraph Product Description Agent"
) as demo:

    gr.Markdown(
        """
# Dynamic LangGraph Product Description Agent

Enter a product name and observe the runtime planning,
dynamic specialist execution, aggregation, and final synthesis.
"""
    )

    with gr.Row():

        product_input = gr.Textbox(
            label="Product Name",
            value="AI-Powered Smart Water Bottle",
            placeholder="Enter product name...",
            scale=4
        )

        run_button = gr.Button(
            "Run Dynamic Agent",
            variant="primary",
            scale=1
        )

    with gr.Tab("Runtime Execution Plan"):

        plan_output = gr.Markdown()

        statistics_output = gr.Markdown()

    with gr.Tab("Basic Description"):

        basic_output = gr.Markdown()

    with gr.Tab("Dynamic Specialist Results"):

        specialist_output = gr.Markdown()

    with gr.Tab("Final Product Description"):

        final_output = gr.Markdown()

    with gr.Tab("Full Process View"):

        full_process_output = gr.Markdown()

    run_button.click(
        fn=run_dynamic_product_agent,
        inputs=product_input,
        outputs=[
            plan_output,
            basic_output,
            specialist_output,
            statistics_output,
            final_output,
            full_process_output
        ]
    )


demo.launch()



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Runtime execution plan: ['features', 'audience', 'seo', 'technical']
Runtime execution plan: ['features', 'audience', 'seo', 'technical']
